<img src="https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/nextia-notebook-branding/assets/courses/c05-header.png" alt="Nextia Learning · Machine Learning: From Problem to Reliable Model" width="100%">

# Forecast bike-share demand

**[C05-M09-L01 · Forecast bike-share demand](https://learning.nextia-ai.com/courses/ml/m09/forecast-bike-share-demand/)** · C05, Machine Learning: From Problem to Reliable Model · Module 9, case study 1 of 7 · [learning.nextia-ai.com](https://learning.nextia-ai.com)

**You will** forecast how many bikes a city bike-share system rents in each hour, from the calendar and the weather, and turn the forecast into a plan. You go from the business question to a tested model, an experiment log, error slices, one final test and a recommendation.

| | |
|---|---|
| **Time** | About 2 hours with the lesson page. The notebook runs in about 1 minute. |
| **Needs** | An internet connection for the setup cells. No account. The first cell installs pandas 3.0.6 and scikit-learn 1.9.1. |
| **You should know** | Modules 1 to 8 of this course, especially [Train, validation and test sets](https://learning.nextia-ai.com/courses/ml/m03/train-validation-and-test-sets/), [Regression metrics](https://learning.nextia-ai.com/courses/ml/m03/regression-metrics/), [Ensembles](https://learning.nextia-ai.com/courses/ml/m05/ensembles/) and [Slice analysis](https://learning.nextia-ai.com/courses/ml/m07/slice-analysis/). |
| **You do not need** | The local ticket-model project. This notebook works in its own folder, `bike-demand/`. |
| **Tested** | 2026-10-07, Python 3.14.6 with pandas 3.0.6, scikit-learn 1.9.1 and matplotlib 3.11.2, in Jupyter on macOS, from a fresh start (*Restart and run all*) |

### How to use this notebook

This notebook has every query, task and check of the lesson. The [lesson page](https://learning.nextia-ai.com/courses/ml/m09/forecast-bike-share-demand/) has the full explanations, the diagrams and the knowledge checks that count toward your certificate. Keep both open.

1. Run the cells in order, from the top. Press **Shift+Enter** to run a cell and move to the next one.
2. At a **Predict** note, write your prediction down before you run the cell.
3. At a **Your turn** note, write your answer in the cell, run it, then run the check cell below it.
4. If you are stuck, open the [solution notebook](https://github.com/usmanahmed99/nextia-learning-labs/blob/main/C05/M09-L01-forecast-bike-share-demand/forecast-bike-share-demand-solution.ipynb).

## Setup: install the packages

Run the next cell. It installs pandas 3.0.6 and scikit-learn 1.9.1, the versions that the course uses, and matplotlib for the charts. It needs an internet connection and takes up to a minute the first time.

You should see `pandas 3.0.6` and `scikit-learn 1.9.1`. If you see other versions, restart the kernel (in Colab: **Runtime › Restart session**; in Kaggle: **Run › Restart & clear cell outputs**), then run this cell again. Restarting is needed when an older version was already loaded.

In [ ]:
%pip install -q --disable-pip-version-check pandas==3.0.6 scikit-learn==1.9.1 matplotlib
import pandas; print("pandas", pandas.__version__)
import sklearn; print("scikit-learn", sklearn.__version__)

## Setup: get the data

Run the next cell. It downloads `bike_sharing_hour.csv.gz` (0.26 MB) from the Nextia labs repository into a folder `bike-demand`, and checks its checksum. It needs an internet connection and no account, and takes a few seconds.

You should see `Ready: bike_sharing_hour.csv.gz, checksum OK.` If you run the cell again, it uses the file that is already there.

If the cell shows an error, read its last line:

- `Could not download`: check your internet connection, then run the cell again. In Kaggle, turn on **Internet** in the notebook settings (it needs a verified phone number).
- `wrong checksum`: delete the folder `bike-demand` (in Colab: the **Files** panel on the left), then run the cell again.

**The data.** Fanaee-T, H. (2013). *Bike Sharing* [Dataset]. UCI Machine Learning Repository. https://doi.org/10.24432/C5W894. Licence: [CC BY 4.0](https://creativecommons.org/licenses/by/4.0/). From the Capital Bikeshare system, Washington, D.C., 2011 and 2012, with weather added by the authors. Cite: Fanaee-T, H. and Gama, J. (2013), Event labeling combining ensemble detectors and background knowledge, *Progress in Artificial Intelligence*, Springer, doi:10.1007/s13748-013-0040-3. We host the file `hour.csv` unchanged, compressed with gzip.

In [ ]:
# Setup: download the case-study data into bike-demand/ and check it.
import hashlib, os, shutil, subprocess, urllib.request
from pathlib import Path

DATA_URL = "https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/C05/case-studies/data/"
NAME, SHA256 = "bike_sharing_hour.csv.gz", "c5ef0c31d5ee006449eaf72f3c2071c68891e82f1f8597074304063e57f1353d"
HERE = Path.cwd()
if Path(".nextia-notebook").exists():  # the cell ran before: start from the parent folder
    os.chdir("..")
    HERE = Path.cwd()
WORK = Path("bike-demand").resolve()
WORK.mkdir(exist_ok=True)
(WORK / ".nextia-notebook").touch()

def sha256(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()

path = WORK / NAME
if not (path.exists() and sha256(path) == SHA256):
    if (HERE / NAME).exists():  # a copy next to the notebook
        shutil.copy(HERE / NAME, path)
    else:
        try:
            urllib.request.urlretrieve(DATA_URL + NAME, path)
        except OSError:  # Python without certificates (python.org on macOS): try curl
            if subprocess.run(["curl", "-fsSL", "-o", str(path), DATA_URL + NAME]).returncode:
                raise SystemExit(f"Could not download {DATA_URL + NAME}. Check your internet connection.")
    if sha256(path) != SHA256:
        raise SystemExit(f"{NAME} has the wrong checksum. Delete the folder bike-demand and run again.")
os.chdir(WORK)  # the experiment log and the saved model go here too
print(f"Ready: {NAME}, checksum OK.")

## Setup: helpers

The next cell defines the check helpers. They tell you if your answer is right, without showing the answer:

- `expect(what, yours, expected)` compares one of your numbers with the expected one, and says "too high" or "too low".
- `expect_hash(what, yours, expected)` compares any result with a hidden fingerprint of the expected one.

It also loads pandas as `pd` and numpy as `np`, and sets pandas to show 4 decimals. Run the cell. It prints nothing.

In [ ]:
# Helpers: check an answer without showing it.
import hashlib, json, warnings
import numpy as np
import pandas as pd
pd.set_option("display.precision", 4)
pd.set_option("display.max_columns", 30)
warnings.filterwarnings("ignore", category=FutureWarning)

def fingerprint(value):
    def norm(v):
        if hasattr(v, "to_dict") and hasattr(v, "columns"):
            return {"columns": [str(c) for c in v.columns], "rows": [norm(list(r)) for r in v.itertuples(index=False)]}
        if hasattr(v, "tolist"):
            return norm(v.tolist())
        if isinstance(v, dict):
            return {str(k): norm(x) for k, x in sorted(v.items(), key=lambda kv: str(kv[0]))}
        if isinstance(v, (list, tuple)):
            return [norm(x) for x in v]
        if isinstance(v, bool) or v is None or isinstance(v, int):
            return v
        if isinstance(v, float):
            return None if v != v else round(v, 4)
        if hasattr(v, "item"):
            return norm(v.item())
        return str(v)
    return hashlib.sha256(json.dumps(norm(value), sort_keys=True).encode()).hexdigest()[:16]

def expect(what, yours, expected, tolerance=1e-6):
    """Compare one number or value with the expected one."""
    if isinstance(yours, (np.integer, np.floating)):
        yours = yours.item()
    if isinstance(expected, float) and isinstance(yours, (int, float)):
        same = abs(yours - expected) <= tolerance * max(1, abs(expected))
    else:
        same = yours == expected
    if same:
        print(f"Check passed: {what} is {yours}.")
    elif isinstance(expected, (int, float)) and isinstance(yours, (int, float)):
        print(f"Not yet: {what} is {yours}, which is too {'high' if yours > expected else 'low'}.")
    else:
        print(f"Not yet: {what} is {yours!r}, which is not the expected value.")

def expect_hash(what, yours, expected):
    """Compare any result with the fingerprint of the expected one."""
    if yours is None or yours is Ellipsis:
        print(f"Not yet: {what} has no value. Write your answer in the cell above, then run it again.")
    elif fingerprint(yours) == expected:
        print(f"Check passed: {what} is right.")
    else:
        print(f"Not yet: {what} is not the expected result. Read the task again, and check each step.")

<img src="https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

## 1. The situation

Amara Okafor runs operations for a city bike-share system. Every evening she plans the next day: how many staff work at the busy stations, and how many vans move bikes from full stations to empty ones. If she plans too little, commuters find empty stations at 8 in the morning. If she plans too much, she pays for vans that wait. She asks Daniel Kim, the analyst, for a forecast of **rentals per hour for tomorrow**. The [lesson page](https://learning.nextia-ai.com/courses/ml/m09/forecast-bike-share-demand/#the-situation) tells the full story.

## 2. The data

Run the next cell. It loads the file and shows the first three hours.

In [ ]:
bikes = pd.read_csv("bike_sharing_hour.csv.gz", parse_dates=["dteday"])
print(bikes.shape)
print(bikes[["dteday", "hr", "workingday", "weathersit", "temp", "casual", "registered", "cnt"]].head(3).to_string())

> **Check.** You should see `(17379, 17)`: 17,379 rows (one per hour) and 17 columns. The first row is midnight on 1 January 2011, with 16 rentals: 3 by casual users and 13 by registered users.

**What this tells us.** One row is one hour of the whole system. `cnt` is the target: all rentals in that hour. `temp` is not in degrees: the authors divided it by 41, so 0.24 is about 10 °C.

Now the first checks: missing values, duplicates, gaps in the calendar and odd codes. Run the cell.

In [ ]:
print("missing values:", int(bikes.isna().sum().sum()))
print("duplicate hours:", int(bikes.duplicated(["dteday", "hr"]).sum()))
calendar_hours = bikes["dteday"].nunique() * 24
print("hours in the calendar:", calendar_hours, "| rows:", len(bikes), "| missing hours:", calendar_hours - len(bikes))
print("weather codes:", bikes["weathersit"].value_counts().sort_index().to_dict())
print("humidity 0:", int((bikes["hum"] == 0).sum()), "| wind speed 0:", int((bikes["windspeed"] == 0).sum()))

> **Check.** You should see `missing values: 0`, `missing hours: 165`, `weather codes: {1: 11413, 2: 4544, 3: 1419, 4: 3}`, `humidity 0: 22` and `wind speed 0: 2180`.

**What this tells us.** No cell is empty, but 165 hours have no row at all. Weather code 4 (heavy rain or snow) appears in only 3 hours, so a model cannot learn much about it. A humidity of 0 is not possible in Washington: those 22 values are sensor gaps. A wind speed of exactly 0 in 2,180 hours is also suspicious: it probably means "not measured".

Which days lose the most hours? Run the cell.

In [ ]:
times = bikes["dteday"] + pd.to_timedelta(bikes["hr"], unit="h")
all_hours = pd.date_range(times.min(), times.max(), freq="h")
missing = pd.Series(all_hours.difference(times))
print(missing.dt.date.value_counts().head(5).to_string())

> **Check.** You should see `2012-10-29` with 23 missing hours at the top, and `2012-10-30` with 13.

**What this tells us.** 29 and 30 October 2012 are the days of Hurricane Sandy: the system closed. The data has no row for an hour that the system was closed, and those days are in our test period.

Now the target, and a check for a leak. Run the cell.

In [ ]:
print(bikes["cnt"].describe().round(1).to_dict())
print("casual + registered == cnt in every row:", bool((bikes["casual"] + bikes["registered"] == bikes["cnt"]).all()))

> **Check.** You should see a median of 142 rentals per hour, a maximum of 977, and `casual + registered == cnt in every row: True`.

**What this tells us.** `casual` and `registered` add up to the target. They are counted in the same hour, so they are not known the evening before. They are a **leak**: never use them as inputs.

## 3. Explore what the decision needs

Amara plans by the hour, so the first question is the shape of a day. Run the cell. It prints five hours of the table; the chart after it shows all 24.

In [ ]:
bikes["day_type"] = np.where(bikes["workingday"] == 1, "working day", "other day")
by_hour = bikes.pivot_table(index="hr", columns="day_type", values="cnt", aggfunc="mean").round(0)
print(by_hour.loc[[4, 8, 13, 17, 22]].to_string())

> **Check.** At 8:00 you should see 477 rentals on a working day and 106 on other days; at 13:00, 198 and 373.

**What this tells us.** Working days have two peaks, at 8:00 and 17:00 to 18:00: commuters. Weekends and holidays have one wide hill around midday. A forecast must know the type of day, not only the hour. Run the next cell for the chart.

In [ ]:
import matplotlib.pyplot as plt

ax = by_hour.plot(figsize=(8, 3.5), marker="o")
ax.set_xlabel("Hour of the day")
ax.set_ylabel("Mean rentals per hour")
plt.show()

Next, the trend over the two years. Run the cell.

In [ ]:
print("mean rentals per hour:", bikes.groupby(bikes["dteday"].dt.year)["cnt"].mean().round(1).to_dict())
monthly = bikes.groupby(bikes["dteday"].dt.to_period("M"))["cnt"].sum()
print("quietest month:", monthly.idxmin(), int(monthly.min()), "| busiest month:", monthly.idxmax(), int(monthly.max()))

> **Check.** You should see `{2011: 143.8, 2012: 234.7}`, the quietest month `2011-01` and the busiest month `2012-09`.

> **Your turn.** By how many percent did the mean rentals per hour grow from 2011 to 2012? Put the answer in `growth_pct`, rounded to 1 decimal, as a Python number. Then run the check cell.

In [ ]:
yearly = bikes.groupby(bikes["dteday"].dt.year)["cnt"].mean()
growth_pct = ...  # the growth in %, rounded to 1 decimal

In [ ]:
expect_hash('growth_pct', growth_pct, '3140121596cecfe9')

**What this tells us.** The system grew fast. A model that learns from the past will be too low in the future unless it can see the growth. Run the next cell for the daily chart.

In [ ]:
daily = bikes.groupby("dteday")["cnt"].sum()
ax = daily.plot(figsize=(9, 3.5), linewidth=0.8)
ax.set_xlabel("Day")
ax.set_ylabel("Rentals per day")
plt.show()

Last, the weather. Run the cell.

In [ ]:
weather = bikes.groupby("weathersit")["cnt"].agg(hours="size", mean_rentals="mean").round(1)
print(weather.to_string())

> **Check.** You should see about 205 rentals per hour in clear weather (code 1), 175 in mist (2) and 112 in light rain or snow (3).

**What this tells us.** Rain cuts rentals by almost half. The forecast must use the weather forecast for tomorrow.

## 4. Split and baseline

The model forecasts the future, so we split by time: **train** to the end of June 2012, **valid** July to September 2012, **test** October to December 2012. The test quarter stays sealed until section 9. Run the cell.

In [ ]:
TARGET = "cnt"
FEATURES = ["hr", "weekday", "workingday", "holiday", "mnth", "yr", "weathersit", "temp", "atemp", "hum", "windspeed"]
train = bikes[bikes["dteday"] < "2012-07-01"]
valid = bikes[(bikes["dteday"] >= "2012-07-01") & (bikes["dteday"] < "2012-10-01")]
test = bikes[bikes["dteday"] >= "2012-10-01"]
print("train", len(train), "| valid", len(valid), "| test", len(test))
print("mean per hour: train", round(train[TARGET].mean(), 1), "| valid", round(valid[TARGET].mean(), 1))

> **Check.** You should see `train 13003 | valid 2208 | test 2168` and a mean of 167.7 rentals per hour in train against 288.4 in valid.

**What this tells us.** Valid is much busier than train: summer, and a bigger system. Every model will feel that.

The next cell defines `report()`: the mean error (positive means the forecast is too low), MAE and RMSE, from [Regression metrics](https://learning.nextia-ai.com/courses/ml/m03/regression-metrics/). Run it. It prints nothing.

In [ ]:
from sklearn.metrics import mean_absolute_error, root_mean_squared_error

def report(real, forecast):
    """Mean error (positive = too low), MAE and RMSE, in bikes per hour."""
    return {"mean error": round(float((real - forecast).mean()), 1),
            "MAE": round(float(mean_absolute_error(real, forecast)), 1),
            "RMSE": round(float(root_mean_squared_error(real, forecast)), 1)}

Two baselines: the mean of train for every hour, and the **hour profile**: the mean of the last 8 weeks of train for each hour of a working day and of other days. A planner can make the profile with a spreadsheet.

> **Predict.** Which baseline has the lower MAE, and by how much? Then run the cell.

In [ ]:
mean_forecast = np.full(len(valid), train[TARGET].mean())
recent = train[train["dteday"] >= "2012-05-06"]  # the last 8 weeks of train
profile = recent.groupby(["workingday", "hr"])[TARGET].mean()
profile_forecast = profile.reindex(pd.MultiIndex.from_frame(valid[["workingday", "hr"]])).to_numpy()
mean_result = report(valid[TARGET], mean_forecast)
profile_result = report(valid[TARGET], profile_forecast)
print("mean of train:", mean_result)
print("hour profile: ", profile_result)

> **Check.** You should see `mean of train: {'mean error': 120.6, 'MAE': 197.8, 'RMSE': 260.7}` and `hour profile:  {'mean error': 14.8, 'MAE': 52.2, 'RMSE': 79.8}`.

**What this tells us.** The hour profile is a strong baseline: 52 bikes off in a typical hour. A model must beat 52.2, not 197.8.

### The experiment log

The next cell defines `log_experiment()` and `show_log()`. They keep one row per experiment in `experiment_log.csv`, as in [Machine learning is iterative](https://learning.nextia-ai.com/courses/ml/m05/machine-learning-is-iterative/#the-experiment-log). Here the columns are the mean error, MAE and RMSE on valid. Run the cell. It prints nothing.

In [ ]:
import datetime
from pathlib import Path

LOG_FILE = Path("experiment_log.csv")
LOG_COLUMNS = ["id", "date", "change", "valid_bias", "valid_mae", "valid_rmse", "decision", "reason"]

def log_experiment(id, change, result, decision, reason):
    """Add one row to the log, or replace the row with the same id."""
    log = pd.read_csv(LOG_FILE) if LOG_FILE.exists() else pd.DataFrame(columns=LOG_COLUMNS)
    row = {"id": id, "date": datetime.date.today().isoformat(), "change": change,
           "valid_bias": result["mean error"], "valid_mae": result["MAE"], "valid_rmse": result["RMSE"],
           "decision": decision, "reason": reason}
    log = pd.concat([log[log["id"] != id], pd.DataFrame([row])], ignore_index=True)
    log.sort_values("id").to_csv(LOG_FILE, index=False)

def show_log():
    log = pd.read_csv(LOG_FILE)
    print(log[["id", "change", "valid_bias", "valid_mae", "valid_rmse", "decision"]].to_string(index=False))

LOG_FILE.unlink(missing_ok=True)  # start a new log

Log the two baselines. Run the cell. You should see rows E01 and E02.

In [ ]:
log_experiment("E01", "Baseline: the mean of train for every hour", mean_result, "baseline", "The floor.")
log_experiment("E02", "Baseline: mean per hour and day type, last 8 weeks of train", profile_result, "baseline",
               "What a planner can do with a spreadsheet; the model must beat it.")
show_log()

## 5. Models

The next cell defines two pipelines. `linear_pipeline()` one-hot encodes the hour, weekday, month and weather, scales the numbers, and adds a linear model, as in [Preprocess with a pipeline](https://learning.nextia-ai.com/courses/ml/m04/preprocess-with-a-pipeline/). `boosting_pipeline()` only keeps the listed columns: trees need no scaling. Both pipelines choose their own columns, so `casual` can never get in by mistake. Run the cell. It prints nothing.

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.linear_model import PoissonRegressor, Ridge
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

CATEGORIES = ["hr", "weekday", "mnth", "weathersit"]
NUMBERS = ["workingday", "holiday", "yr", "temp", "atemp", "hum", "windspeed"]

def linear_pipeline(model, categories=CATEGORIES):
    """One-hot codes for the categories, scaled numbers, then a linear model."""
    prepare = ColumnTransformer([("categories", OneHotEncoder(handle_unknown="ignore"), categories),
                                 ("numbers", StandardScaler(), NUMBERS)])
    return Pipeline([("prepare", prepare), ("model", model)])

def boosting_pipeline(model, features=FEATURES):
    """Keep only the listed columns, then the model. Trees need no scaling."""
    select = ColumnTransformer([("features", "passthrough", features)])
    return Pipeline([("select", select), ("model", model)])

First, ridge regression: a linear model with a small penalty. Run the cell.

In [ ]:
pipeline = linear_pipeline(Ridge(alpha=1.0)).fit(train, train[TARGET])
ridge_forecast = pipeline.predict(valid)
ridge_result = report(valid[TARGET], ridge_forecast)
print(ridge_result)

> **Check.** You should see `{'mean error': 17.5, 'MAE': 111.4, 'RMSE': 146.1}`: twice as bad as the hour profile.

Why? Look at four hours of a working day. Run the cell.

In [ ]:
check = valid.assign(forecast=ridge_forecast)
hours = check[check["workingday"] == 1].groupby("hr")[[TARGET, "forecast"]].mean().round(0)
print(hours.loc[[3, 8, 13, 17]].to_string())

> **Check.** At 3:00 you should see 7 real rentals and a forecast of 109; at 17:00, 813 and 518.

**What this tells us.** A linear model **adds** effects: "+ so many bikes for July, + so many for sun". But rain does not take the same 50 bikes from every hour: it takes half of them. Rentals **multiply**.

**Poisson regression** models the logarithm of the count, so its effects multiply: "rain × 0.6". Its forecasts are never below 0. Run the cell.

In [ ]:
pipeline = linear_pipeline(PoissonRegressor(alpha=1e-4, max_iter=1000)).fit(train, train[TARGET])
poisson_result = report(valid[TARGET], pipeline.predict(valid))
print(poisson_result)

> **Check.** You should see an MAE of 93.0, and a mean error of −28.0 (too high).

Better, but the model still uses one daily shape for every day. Add a category `hour_type` with 48 values: each hour of a working day, and each hour of other days. Run the cell.

In [ ]:
def add_hour_type(part):
    """One category per hour of a working day and per hour of other days: 48 values."""
    return part.assign(hour_type=part["workingday"].astype(str) + "-" + part["hr"].astype(str))

train, valid, test = add_hour_type(train), add_hour_type(valid), add_hour_type(test)
pipeline = linear_pipeline(PoissonRegressor(alpha=1e-4, max_iter=1000), CATEGORIES + ["hour_type"])
pipeline.fit(train, train[TARGET])
hour_type_result = report(valid[TARGET], pipeline.predict(valid))
print(hour_type_result)

> **Check.** You should see an MAE of 54.8: the shape of the day was the missing piece.

Log the three linear models. Run the cell.

In [ ]:
log_experiment("E03", "Ridge, one-hot hour, weekday, month, weather", ridge_result, "reject",
               "Far worse than the hour profile: it adds effects, but rentals multiply.")
log_experiment("E04", "Poisson regression, the same inputs", poisson_result, "reject",
               "Better than ridge, still far worse than the hour profile.")
log_experiment("E05", "E04 + hour_type (hour x working day)", hour_type_result, "reject",
               "The shape of the day was the missing piece; still worse than the profile.")
show_log()

Now gradient boosting, from [Ensembles](https://learning.nextia-ai.com/courses/ml/m05/ensembles/): many small trees, each one fixing the errors of the trees before it. Trees find the hour-by-day-type shape alone. We try two **losses**: squared error (the default) and the Poisson loss, made for counts.

> **Predict.** Will boosting beat the hour profile's 52.2? Which loss forecasts fewer than 0 bikes? Then run the cell.

In [ ]:
boosting = boosting_pipeline(HistGradientBoostingRegressor(random_state=0)).fit(train, train[TARGET])
forecast = boosting.predict(valid)
squared_result = report(valid[TARGET], forecast)
print("squared error:", squared_result, "| hours below 0:", int((forecast < 0).sum()))
boosting = boosting_pipeline(HistGradientBoostingRegressor(loss="poisson", random_state=0)).fit(train, train[TARGET])
forecast = boosting.predict(valid)
boosting_result = report(valid[TARGET], forecast)
print("Poisson loss: ", boosting_result, "| hours below 0:", int((forecast < 0).sum()))

> **Check.** You should see `squared error: {'mean error': 20.1, 'MAE': 40.9, 'RMSE': 60.8} | hours below 0: 10` and `Poisson loss:  {'mean error': 21.4, 'MAE': 40.3, 'RMSE': 62.4} | hours below 0: 0`.

**What this tells us.** Both beat the profile by about 12 bikes. The squared-error model forecasts fewer than 0 bikes in 10 night hours, which makes no sense to Amara. Both are too low by about 20 bikes per hour: valid is busier than anything in train. Log them. Run the cell.

In [ ]:
log_experiment("E06", "Gradient boosting, squared error, defaults", squared_result, "reject",
               "Good, but it forecasts fewer than 0 bikes in some hours.")
log_experiment("E07", "Gradient boosting, Poisson loss, defaults", boosting_result, "keep",
               "Best MAE so far, never below 0; 12 bikes better than the profile.")
show_log()

### Your turn: past rentals as inputs

Daniel suggests two more inputs: the rentals in the same hour **2 days before** and **7 days before**. Both are known the evening before. (The same hour **1 day** before is not: at 20:00 today, the rest of today has not happened yet.)

The next cell defines `add_past_rentals(part, days)`, which adds a column `rentals_<days>d_before`. Run it. It prints nothing.

In [ ]:
def add_past_rentals(part, days):
    """Rentals in the same hour `days` days earlier (empty if that hour is missing)."""
    times = bikes["dteday"] + pd.to_timedelta(bikes["hr"], unit="h")
    rentals = pd.Series(bikes["cnt"].to_numpy(), index=times)
    part_times = part["dteday"] + pd.to_timedelta(part["hr"], unit="h")
    past = rentals.reindex(part_times - pd.Timedelta(days=days)).to_numpy()
    return part.assign(**{f"rentals_{days}d_before": past})

> **Your turn.** Add both columns to `train` and `valid` (keep them in a list `with_past`). Fit `boosting_pipeline()` with a Poisson `HistGradientBoostingRegressor(loss="poisson", random_state=0)` and the features `FEATURES + ["rentals_2d_before", "rentals_7d_before"]`. Put the `report()` on valid in `past_result`, and log it as E08 with your decision. Then run the check cell.
>
> **Predict** first: will the past rentals help?

In [ ]:
with_past = ...  # [train with the two columns, valid with the two columns]
past_result = ...  # report() of the new model on valid
# log_experiment("E08", "E07 + rentals in the same hour 2 and 7 days before", past_result, ..., ...)

In [ ]:
expect_hash('past_result', past_result, '9e0e77d996497a1b')

The past rentals make the forecast **worse** (MAE 45.3 against 40.3). The rentals of two days ago carry that day's weather and events, which tomorrow does not share. A rejected idea is still a result: keep it in the log.

### Tune one setting, and check it twice

A slower boosting (learning rate 0.05, 400 trees) is the last experiment. A gain of 1 or 2 bikes on one quarter can be luck, so we also check it on April to June 2012, learning only from the months before. Run the cell.

In [ ]:
slower = HistGradientBoostingRegressor(loss="poisson", learning_rate=0.05, max_iter=400, random_state=0)
slower_result = report(valid[TARGET], boosting_pipeline(slower).fit(train, train[TARGET]).predict(valid))
print("E09 on valid:", slower_result)
# Check the choice on an earlier quarter too: learn before April 2012, forecast April-June 2012.
early, spring = train[train["dteday"] < "2012-04-01"], train[train["dteday"] >= "2012-04-01"]
for name, model in [("E07", HistGradientBoostingRegressor(loss="poisson", random_state=0)), ("E09", slower)]:
    print(name, "on April-June 2012:", report(spring[TARGET], boosting_pipeline(model).fit(early, early[TARGET]).predict(spring)))

> **Check.** You should see `E09 on valid: {'mean error': 21.6, 'MAE': 38.7, 'RMSE': 61.0}`, then MAE 39.3 for E07 and 37.6 for E09 on April to June 2012.

**What this tells us.** E09 wins on both quarters, so it is the final setting. Run the cell to log it.

In [ ]:
log_experiment("E09", "E07 with learning_rate 0.05 and 400 trees", slower_result, "keep",
               "Better than E07 on valid and on April-June 2012; the final settings.")
show_log()

## 6. Failure case: the random split that lies

Lena, a new analyst on the team, would split the hours at random, as in many tutorials. Run the cell. It also tries the leak `casual`.

In [ ]:
from sklearn.model_selection import train_test_split

shuffled_train, shuffled_test = train_test_split(bikes, test_size=0.2, random_state=0)
model = HistGradientBoostingRegressor(loss="poisson", random_state=0)
pipeline = boosting_pipeline(model).fit(shuffled_train, shuffled_train[TARGET])
print("random split:", report(shuffled_test[TARGET], pipeline.predict(shuffled_test)))
pipeline = boosting_pipeline(model, FEATURES + ["casual"]).fit(shuffled_train, shuffled_train[TARGET])
print("random split + casual:", report(shuffled_test[TARGET], pipeline.predict(shuffled_test)))

> **Check.** You should see `random split: {'mean error': 0.7, 'MAE': 27.3, 'RMSE': 45.6}` and `random split + casual: {'mean error': 0.8, 'MAE': 21.4, 'RMSE': 34.7}`.

**What this tells us.** A random split gives MAE 27, much better than the honest 38.7. It lies: the model sees the hours just before and after each test hour, and the level of the same weeks. Amara never has tomorrow's neighbours. With `casual`, the score looks even better, and the model is useless the evening before.

## 7. Metric and operating point

The chosen model on valid. Run the cell.

In [ ]:
chosen = HistGradientBoostingRegressor(loss="poisson", learning_rate=0.05, max_iter=400, random_state=0)
pipeline = boosting_pipeline(chosen).fit(train, train[TARGET])
valid_forecast = pipeline.predict(valid)
print(report(valid[TARGET], valid_forecast))

> **Check.** You should see `{'mean error': 21.6, 'MAE': 38.7, 'RMSE': 61.0}`.

Amara does not plan with the forecast itself: it is too low on average, and an empty station costs more than a spare bike. She plans with **forecast × a plan factor**, chosen on valid so that the plan covers 9 of 10 busy hours (a forecast of 100 bikes or more). Run the cell.

In [ ]:
busy = valid_forecast >= 100
ratio = valid[TARGET].to_numpy()[busy] / valid_forecast[busy]
plan_factor = round(float(np.quantile(ratio, 0.9)), 2)
print("busy hours in valid:", int(busy.sum()), "| plan factor:", plan_factor)
covered = (valid[TARGET].to_numpy()[busy] <= plan_factor * valid_forecast[busy]).mean()
print("busy hours covered by forecast x factor:", round(float(covered), 3))

> **Check.** You should see `busy hours in valid: 1618 | plan factor: 1.26` and a coverage of `0.891`.

## 8. Errors and slices (on valid)

Where does the model fail? First by time of day. Run the cell.

In [ ]:
RUSH = [7, 8, 9, 16, 17, 18, 19]

def time_of_day(part):
    rush = (part["workingday"] == 1) & part["hr"].isin(RUSH)
    day = part["hr"].between(7, 19)
    return np.select([rush, day], ["rush hour", "other day hour"], "night (20-6)")

def slice_table(part, forecast, by):
    errors = part.assign(error=part[TARGET] - forecast, abs_error=(part[TARGET] - forecast).abs())
    return errors.groupby(by).agg(hours=("error", "size"), mean_rentals=(TARGET, "mean"),
                                  MAE=("abs_error", "mean"), mean_error=("error", "mean")).round(1)

print(slice_table(valid, valid_forecast, time_of_day(valid)).to_string())

> **Check.** In rush hours you should see 441 hours, a mean of 586.0 rentals, an MAE of 74.8 and a mean error of 57.7.

**What this tells us.** The rush hours have the largest errors, and the model is too low there by 58 bikes on average: exactly when stations run empty.

> **Your turn.** Make the same table by weather code: call `slice_table()` with `valid["weathersit"]`. Put the result in `weather_slices`, then run the check cell.

In [ ]:
weather_slices = ...  # slice_table() of valid, by weathersit

In [ ]:
expect_hash('weather_slices', weather_slices, '16080adbf2033018')

In light rain (code 3), the MAE is more than twice the MAE in clear weather. Rain is hard to forecast by the hour, and a shower at 17:00 empties the commute.

Last, the worst days. Run the cell.

In [ ]:
days = valid.assign(abs_error=np.abs(valid[TARGET] - valid_forecast), forecast=valid_forecast)
worst = days.groupby("dteday").agg(rentals=(TARGET, "sum"), forecast=("forecast", "sum"), MAE=("abs_error", "mean"))
print(worst.sort_values("MAE", ascending=False).head(4).round(0).to_string())

> **Check.** At the top you should see `2012-07-04`, Independence Day: 7,403 rentals against a forecast of 5,948.

**What this tells us.** Holidays with events (fireworks on the National Mall) are not ordinary holidays. The model only knows "holiday: yes".

## 9. The final test, once

The plan was fixed before this cell: refit E09 on train and valid together (the most recent data), then forecast the test quarter once. Run the cell.

In [ ]:
train_and_valid = pd.concat([train, valid])
final = boosting_pipeline(chosen).fit(train_and_valid, train_and_valid[TARGET])
test_forecast = final.predict(test)
final_result = report(test[TARGET], test_forecast)
print("final model on test:", final_result)

> **Check.** You should see `final model on test: {'mean error': 15.3, 'MAE': 44.5, 'RMSE': 69.6}`.

The same two baselines, made the same way from the data before October. Run the cell.

In [ ]:
recent = train_and_valid[train_and_valid["dteday"] >= "2012-08-06"]  # the last 8 weeks before test
profile = recent.groupby(["workingday", "hr"])[TARGET].mean()
test_profile = profile.reindex(pd.MultiIndex.from_frame(test[["workingday", "hr"]])).to_numpy()
print("mean of train and valid:", report(test[TARGET], np.full(len(test), train_and_valid[TARGET].mean())))
print("hour profile:           ", report(test[TARGET], test_profile))

> **Check.** You should see MAE 156.9 for the mean and 94.1 for the hour profile.

**What this tells us.** The model is off by about 45 bikes per hour in autumn: worse than on valid (38.7), but less than half the error of the hour profile, whose summer shape is wrong in the autumn. Now the slices of the test quarter: we report them, we do not tune on them. Run the cell.

In [ ]:
print(slice_table(test, test_forecast, time_of_day(test)).to_string())
sandy = test["dteday"].isin(pd.to_datetime(["2012-10-29", "2012-10-30"]))
print("Hurricane Sandy, 29-30 Oct:", int(sandy.sum()), "hours, MAE",
      round(float(np.abs(test[TARGET] - test_forecast)[sandy].mean()), 1))
by_month = slice_table(test, test_forecast, test["dteday"].dt.month)
print(by_month[["hours", "mean_rentals", "MAE", "mean_error"]].to_string())

> **Check.** You should see a rush-hour MAE of 88.7 with a mean error of 39.9, and `Hurricane Sandy, 29-30 Oct: 12 hours, MAE 79.8`.

Does the plan factor still cover 9 of 10 busy hours? Run the cell.

In [ ]:
busy = test_forecast >= 100
covered = (test[TARGET].to_numpy()[busy] <= plan_factor * test_forecast[busy]).mean()
print("busy test hours:", int(busy.sum()), "| covered by forecast x", plan_factor, ":", round(float(covered), 3))

> **Check.** You should see a coverage of `0.788`.

**What this tells us.** The factor from the summer covers only about 79% of busy autumn hours, not 90%. A factor must be checked and re-chosen as the season changes.

## 10. Save the model and the log

Run the cell. It saves the final pipeline to `bike_demand_model.joblib` and shows the full log, as in [Package artifacts](https://learning.nextia-ai.com/courses/ml/m08/package-artifacts/).

In [ ]:
import joblib

joblib.dump(final, "bike_demand_model.joblib")
show_log()

> **Check.** You should see nine rows, E01 to E09.

## Next

You have taken a forecast from the question to one honest test. The [lesson page](https://learning.nextia-ai.com/courses/ml/m09/forecast-bike-share-demand/) has the recommendations for Amara, the limits, the model card, how to make this a portfolio piece, and the knowledge checks.

<img src="https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

**Next lesson:** [Estimate house prices](https://learning.nextia-ai.com/courses/ml/m09/estimate-house-prices/) · **This lesson:** [read it on the site](https://learning.nextia-ai.com/courses/ml/m09/forecast-bike-share-demand/)

<img src="https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/nextia-notebook-branding/assets/notebook-footer-light.png" alt="Nextia Learning · learning.nextia-ai.com" width="100%">

Free, practical AI courses at [learning.nextia-ai.com](https://learning.nextia-ai.com). The data is real: UCI Bike Sharing (Fanaee-T, 2013), CC BY 4.0.